In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver() #checkpointer to save the state of the workflow

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}} #thread id to save the state of the workflow
workflow.invoke({'topic':'KFC'}, config=config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'KFC',
 'joke': [{'type': 'text',
   'text': 'I asked the cashier at KFC, "What’s the secret to making the chicken so crispy and delicious?"\n\nShe leaned in and whispered, "Honestly? It all starts with a chicken having a really, really bad day."',
   'extras': {'signature': 'EvocCvccARFNMg8b41h7UI8NlKT/dfncktBc+28CxlgiiW7L8ngsX2HCRPIvpE82HP+AkKdYXobWv5KqNGu/fWq4K6tC+/ox33dcr2VL1+didXt51TvzJ9SgwNlrnQwsdu6BlW9rUDceIL9F6WIfHU5XAEoQSJTTXIznOVp9cPE/gJsXvWvtXuCbc2SELnJbgtbCNy06VkPBTmTadDdVenzKe+bBD3icMxY0ZHlZFty6baTrpxEAvNYrlA5N6VPLEixi5OZN3Mcxpt70UKcxkZjUtLJkuhOikFhvGl1sQN6tt1o6J/GB2rshal5+ls3U9EF75zxtmv3dvbwDiQ16nQ5pAoNUQnIcShypYgrtIGaumeU62HUufYXe+rfApXCuN7COFm9CdLFdqYCE1SaOExiOX3LKRi4ERBFkyfA5Q7kYDT4o8t5QuOmVB90wcKcy8O8FUvrzf1YvC073cn6jemhs+SLAyFnIf6fyjPqqimcWt57CxIOkp2cHvu7yczeJ6wWiHYL+dl8PhVSeDoDDYjqWnL21BT2EKD4YquFZQ4muFboSIqyMxZ2VDNYnOlY/IcbU5rUI8iSPMTbGq4aKn3w3ohA9WE/zLWGue+w2u6vL/zDyt5ZPde9KNc06+SKXYoSnrBhGj1hJF/rEjSTIg6wsxzVxUQbkQwRAa1glFlZLJDekn+jt8RBxw+IYYsuqI96s0iBs9

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'KFC', 'joke': [{'type': 'text', 'text': 'I asked the cashier at KFC, "What’s the secret to making the chicken so crispy and delicious?"\n\nShe leaned in and whispered, "Honestly? It all starts with a chicken having a really, really bad day."', 'extras': {'signature': 'EvocCvccARFNMg8b41h7UI8NlKT/dfncktBc+28CxlgiiW7L8ngsX2HCRPIvpE82HP+AkKdYXobWv5KqNGu/fWq4K6tC+/ox33dcr2VL1+didXt51TvzJ9SgwNlrnQwsdu6BlW9rUDceIL9F6WIfHU5XAEoQSJTTXIznOVp9cPE/gJsXvWvtXuCbc2SELnJbgtbCNy06VkPBTmTadDdVenzKe+bBD3icMxY0ZHlZFty6baTrpxEAvNYrlA5N6VPLEixi5OZN3Mcxpt70UKcxkZjUtLJkuhOikFhvGl1sQN6tt1o6J/GB2rshal5+ls3U9EF75zxtmv3dvbwDiQ16nQ5pAoNUQnIcShypYgrtIGaumeU62HUufYXe+rfApXCuN7COFm9CdLFdqYCE1SaOExiOX3LKRi4ERBFkyfA5Q7kYDT4o8t5QuOmVB90wcKcy8O8FUvrzf1YvC073cn6jemhs+SLAyFnIf6fyjPqqimcWt57CxIOkp2cHvu7yczeJ6wWiHYL+dl8PhVSeDoDDYjqWnL21BT2EKD4YquFZQ4muFboSIqyMxZ2VDNYnOlY/IcbU5rUI8iSPMTbGq4aKn3w3ohA9WE/zLWGue+w2u6vL/zDyt5ZPde9KNc06+SKXYoSnrBhGj1hJF/rEjSTIg6wsxzVxUQbkQwRAa1glFlZLJDekn+jt8RBxw+I

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'KFC', 'joke': [{'type': 'text', 'text': 'I asked the cashier at KFC, "What’s the secret to making the chicken so crispy and delicious?"\n\nShe leaned in and whispered, "Honestly? It all starts with a chicken having a really, really bad day."', 'extras': {'signature': 'EvocCvccARFNMg8b41h7UI8NlKT/dfncktBc+28CxlgiiW7L8ngsX2HCRPIvpE82HP+AkKdYXobWv5KqNGu/fWq4K6tC+/ox33dcr2VL1+didXt51TvzJ9SgwNlrnQwsdu6BlW9rUDceIL9F6WIfHU5XAEoQSJTTXIznOVp9cPE/gJsXvWvtXuCbc2SELnJbgtbCNy06VkPBTmTadDdVenzKe+bBD3icMxY0ZHlZFty6baTrpxEAvNYrlA5N6VPLEixi5OZN3Mcxpt70UKcxkZjUtLJkuhOikFhvGl1sQN6tt1o6J/GB2rshal5+ls3U9EF75zxtmv3dvbwDiQ16nQ5pAoNUQnIcShypYgrtIGaumeU62HUufYXe+rfApXCuN7COFm9CdLFdqYCE1SaOExiOX3LKRi4ERBFkyfA5Q7kYDT4o8t5QuOmVB90wcKcy8O8FUvrzf1YvC073cn6jemhs+SLAyFnIf6fyjPqqimcWt57CxIOkp2cHvu7yczeJ6wWiHYL+dl8PhVSeDoDDYjqWnL21BT2EKD4YquFZQ4muFboSIqyMxZ2VDNYnOlY/IcbU5rUI8iSPMTbGq4aKn3w3ohA9WE/zLWGue+w2u6vL/zDyt5ZPde9KNc06+SKXYoSnrBhGj1hJF/rEjSTIg6wsxzVxUQbkQwRAa1glFlZLJDekn+jt8RBxw+

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': "What do you call a fake noodle?\n\nAn **impasta**! \n\n*(I'd tell you another one, but it's a bit too saucy.)*",
   'extras': {'signature': 'EtEZCs4ZARFNMg/jbKp707aA4kL89aXu3b4wOAOfuPNvGQx9ye7FtFp9bHTcqtyE6jNM9ohnzDmfsM5e9b3i+ioIu3+c9gBClxSqqlr+O1WyIf/WlmArzWuEtfQq2GXe+lpV15s3svE3IxpVZDF8w16MoLwlgVQP2kSiCKrQAgQEqEaElqOO53g5UZnJud8GpY1El805QooDWtStBwiSlJO7YOcDcsItVEf+vrW89vNcxnKDTgsyV9kAwAyttO8wi7dlilR7lqAzQOP6AmBv21K60AmuPl7IScXxJRMJie1LBvjah5KgVPio9oxYqu3mG69WUMg/mcTn97ABKey5hmaJMnbT+0mjJtQItFCXpZdlxk1bUN7g+HmXY5T7Nk+OW7ekPhGVDRSHk98+1aNbTRKdrdKymn7KByRds/rV/GC/8hWxEbDtIwA4uGKqd37B3L506rmQ6v08VG2HdgEWbDpBHTaSQsg0RXqe4+dG/Q8UdERh2VPvNs0i3DKfZhBw9hqR5XAA/C5MDRu5CHFy1uq0vtYIL40ewKvfZLCiEyJCRJTT6g93WjyzlVmW1NVlvB+JJXFI6NEDt0IC4FWxoL/dmAAIbE7jn5efszzy2c6kOxb0S3l1+A9L3sm9TxVpYq1fP4wgx4cAPBfTrxrPjnDHKgqImZ1YZigMBHgQkxur1BmyOjVs5gx81DeBfDEFKirTDwvB0MdLJ9XhTPzhZe5OQ4tCBeuOhF40kF3dVi5sM0Gfnwft8MDRR4vHF9ig+fVe5P9q1F6M9J9tDmCZZvXRYgGXKhKBU

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'KFC', 'joke': [{'type': 'text', 'text': 'I asked the cashier at KFC, "What’s the secret to making the chicken so crispy and delicious?"\n\nShe leaned in and whispered, "Honestly? It all starts with a chicken having a really, really bad day."', 'extras': {'signature': 'EvocCvccARFNMg8b41h7UI8NlKT/dfncktBc+28CxlgiiW7L8ngsX2HCRPIvpE82HP+AkKdYXobWv5KqNGu/fWq4K6tC+/ox33dcr2VL1+didXt51TvzJ9SgwNlrnQwsdu6BlW9rUDceIL9F6WIfHU5XAEoQSJTTXIznOVp9cPE/gJsXvWvtXuCbc2SELnJbgtbCNy06VkPBTmTadDdVenzKe+bBD3icMxY0ZHlZFty6baTrpxEAvNYrlA5N6VPLEixi5OZN3Mcxpt70UKcxkZjUtLJkuhOikFhvGl1sQN6tt1o6J/GB2rshal5+ls3U9EF75zxtmv3dvbwDiQ16nQ5pAoNUQnIcShypYgrtIGaumeU62HUufYXe+rfApXCuN7COFm9CdLFdqYCE1SaOExiOX3LKRi4ERBFkyfA5Q7kYDT4o8t5QuOmVB90wcKcy8O8FUvrzf1YvC073cn6jemhs+SLAyFnIf6fyjPqqimcWt57CxIOkp2cHvu7yczeJ6wWiHYL+dl8PhVSeDoDDYjqWnL21BT2EKD4YquFZQ4muFboSIqyMxZ2VDNYnOlY/IcbU5rUI8iSPMTbGq4aKn3w3ohA9WE/zLWGue+w2u6vL/zDyt5ZPde9KNc06+SKXYoSnrBhGj1hJF/rEjSTIg6wsxzVxUQbkQwRAa1glFlZLJDekn+jt8RBxw+I

In [14]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': "What do you call a fake noodle?\n\nAn **impasta**! \n\n*(I'd tell you another one, but it's a bit too saucy.)*", 'extras': {'signature': 'EtEZCs4ZARFNMg/jbKp707aA4kL89aXu3b4wOAOfuPNvGQx9ye7FtFp9bHTcqtyE6jNM9ohnzDmfsM5e9b3i+ioIu3+c9gBClxSqqlr+O1WyIf/WlmArzWuEtfQq2GXe+lpV15s3svE3IxpVZDF8w16MoLwlgVQP2kSiCKrQAgQEqEaElqOO53g5UZnJud8GpY1El805QooDWtStBwiSlJO7YOcDcsItVEf+vrW89vNcxnKDTgsyV9kAwAyttO8wi7dlilR7lqAzQOP6AmBv21K60AmuPl7IScXxJRMJie1LBvjah5KgVPio9oxYqu3mG69WUMg/mcTn97ABKey5hmaJMnbT+0mjJtQItFCXpZdlxk1bUN7g+HmXY5T7Nk+OW7ekPhGVDRSHk98+1aNbTRKdrdKymn7KByRds/rV/GC/8hWxEbDtIwA4uGKqd37B3L506rmQ6v08VG2HdgEWbDpBHTaSQsg0RXqe4+dG/Q8UdERh2VPvNs0i3DKfZhBw9hqR5XAA/C5MDRu5CHFy1uq0vtYIL40ewKvfZLCiEyJCRJTT6g93WjyzlVmW1NVlvB+JJXFI6NEDt0IC4FWxoL/dmAAIbE7jn5efszzy2c6kOxb0S3l1+A9L3sm9TxVpYq1fP4wgx4cAPBfTrxrPjnDHKgqImZ1YZigMBHgQkxur1BmyOjVs5gx81DeBfDEFKirTDwvB0MdLJ9XhTPzhZe5OQ4tCBeuOhF40kF3dVi5sM0Gfnwft8MDRR4vHF9ig+fVe5P9q1F6M9J9tDm

### Time Travel

In [16]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1b2811-5371-69a2-8000-3fa2f25a47e1"}})

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1b2811-5371-69a2-8000-3fa2f25a47e1'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-17T10:18:10.165392+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b2811-536e-6fb5-bfff-324141a0021e'}}, tasks=(PregelTask(id='43b49a32-b4ba-65dd-f035-50f55030460a', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': [{'type': 'text', 'text': "What do you call a fake noodle?\n\nAn **impasta**! \n\n*(I'd tell you another one, but it's a bit too saucy.)*", 'extras': {'signature': 'EtEZCs4ZARFNMg/jbKp707aA4kL89aXu3b4wOAOfuPNvGQx9ye7FtFp9bHTcqtyE6jNM9ohnzDmfsM5e9b3i+ioIu3+c9gBClxSqqlr+O1WyIf/WlmArzWuEtfQq2GXe+lpV15s3svE3IxpVZDF8w16MoLwlgVQP2kSiCKrQAgQEqEaElqOO53g5UZnJud8GpY1El805QooDWtStBwiSlJO7YOcDcsItVEf+vrW89vNcxnKDTgsyV9kAwAyttO8w

In [17]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1b2811-5371-69a2-8000-3fa2f25a47e1"}})

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': '**Q:** What do you call a fake noodle?\n\n**A:** An **impasta**! \n\n***\n\n*Bonus:*\n\n**Q:** Why did the spaghetti break up with the sauce?\n**A:** Because their relationship was getting way too **strained**!',
   'extras': {'signature': 'EusgCuggARFNMg92hucPeJozxdVuP2dHcD/pUc3sivOwqN8Q33s0ALk4tEZ4vSDGIGH+nUYTbE2MwBJCQ2hO8fhZZmQ8Z/eG2mvQnPDXI78tfSYbubSJ8nhc2fnUi0Azzp1HAOrO4K9Mgy1iNr3LNI4vrXVkiNkEpW0joAmb2eWZU62KRkOIfCB/ABPhGAu+PH4CDiOat7b07ZaJFVv+mHRUxmTgqLi9ec9yGiatUIwqpwxgFm9s1qBIb6TVSzOx+YW9qQWPaqZhcz4KHn0rPy+iTB4DDg/6cEfhthcnHyTbTDEBjSEmL8+yKJmni8a+B+LTq2a+DpCgwguTvKTa4GnYZuyDU8gL3uaO1fIisuCEoNgi5ymvflkE5SQ+GWN4GsxsAqHcm9kpPbMEi75L5uu57PsouZbqL9e+dRixoKry08AbBY8bhPgGxopg+gpgLNxS2w/eOLyV1PxPl2SQUuhY5GCk6p/S+Rak/HQm7o4eB8krHoWX7nJLhUyYk3l9lVvIacMdxiDHGjBOJWDxoZSSuStsuBi+DtXbwGSyk53afHv7TpYO8lAQGYe+PWLdWYQtk5wKgaIs7sC+tRp205ymh/GODbhKQmNai1Zb59qc8z/5RwTYDE2nvG56ewESpWQoh6/w6vJ2JXMGyGtVz1iC/NVu2TXBFGLSTKkb4o7rzS3VbSTBkEAkL/43I

In [18]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': '**Q:** What do you call a fake noodle?\n\n**A:** An **impasta**! \n\n***\n\n*Bonus:*\n\n**Q:** Why did the spaghetti break up with the sauce?\n**A:** Because their relationship was getting way too **strained**!', 'extras': {'signature': 'EusgCuggARFNMg92hucPeJozxdVuP2dHcD/pUc3sivOwqN8Q33s0ALk4tEZ4vSDGIGH+nUYTbE2MwBJCQ2hO8fhZZmQ8Z/eG2mvQnPDXI78tfSYbubSJ8nhc2fnUi0Azzp1HAOrO4K9Mgy1iNr3LNI4vrXVkiNkEpW0joAmb2eWZU62KRkOIfCB/ABPhGAu+PH4CDiOat7b07ZaJFVv+mHRUxmTgqLi9ec9yGiatUIwqpwxgFm9s1qBIb6TVSzOx+YW9qQWPaqZhcz4KHn0rPy+iTB4DDg/6cEfhthcnHyTbTDEBjSEmL8+yKJmni8a+B+LTq2a+DpCgwguTvKTa4GnYZuyDU8gL3uaO1fIisuCEoNgi5ymvflkE5SQ+GWN4GsxsAqHcm9kpPbMEi75L5uu57PsouZbqL9e+dRixoKry08AbBY8bhPgGxopg+gpgLNxS2w/eOLyV1PxPl2SQUuhY5GCk6p/S+Rak/HQm7o4eB8krHoWX7nJLhUyYk3l9lVvIacMdxiDHGjBOJWDxoZSSuStsuBi+DtXbwGSyk53afHv7TpYO8lAQGYe+PWLdWYQtk5wKgaIs7sC+tRp205ymh/GODbhKQmNai1Zb59qc8z/5RwTYDE2nvG56ewESpWQoh6/w6vJ2JXMGyGtVz1iC/NVu2TXBFGLSTKkb4o7rzS

#### Updating State

In [20]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f1b2811-5371-69a2-8000-3fa2f25a47e1", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b2834-7b09-6dde-8001-7c09f8e5de53'}}

In [21]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b2834-7b09-6dde-8001-7c09f8e5de53'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-17T10:33:53.841302+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b2811-5371-69a2-8000-3fa2f25a47e1'}}, tasks=(PregelTask(id='425002a1-4c6c-d322-e7be-d1fed3edd07b', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b2834-384d-6070-8001-83eabf08a7cf'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-17T10:33:46.843352+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id

In [23]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f1b2834-7b09-6dde-8001-7c09f8e5de53"}})

{'topic': 'samosa',
 'joke': [{'type': 'text',
   'text': 'Why did the samosa go to therapy?\n\nBecause it was under too much pressure, felt totally **deep-fried**, and was keeping way too many **fillings** inside! \n\n***\n\n**Bonus short pun:** \nI love you **samosa** it hurts! 🥟❤️',
   'extras': {'signature': 'Ep4bCpsbARFNMg8eLVy+J3Uxiyv/7wMHt5ICPRsHcR3atIGhTLdJSCCso8npax+9DCz+HCRDCYGrsNVeLCa4cz5xAS4RXxjW7wiA97eWuB67qvyCIS3fgV1z4fUKjjhdZ8J9/PLNX8YYkMGvp7QquZRnVxjpAumf2SpyhN3GxPC6FG7r2xdAllV5F8H0KhqqtAeCf9fqMhMQ2v6w8yEmcBw6VC6KgpRkR7L03zTArkG+68mfLv9RVwlfNYaH/y1g56usTMigiW4hcKIyJeN3fwqfsVRQ7dqWS7bXMrueuTh5yygr0qMOHibQyuyUC4a168mAPsofPVMPB3FIjYOOt3TpUjNLByPzBeqy1IbmdfFdFGtDhpfIKrQ9Y3hkeL9vNGfK+DznTfVJidNQRNL1N94zdvc/ICczEk0n9c84Y3sIjD5esPGHzE04Wblg/9K/X0TR2BfzGVydrKQeVmQiXa2dKoUHoyIAXyQG9ZbickD5r4Y96XWH33evo6bM0rj9am2hA5jbCBt8Bb4ZVPzvMDsN3Iyy+7TYrfZI6JtlLlnR1fjgdszeo4f83vUjNXAhvUHsE27px/cs9QcpyrWKHNU4SQdJrwxWdkx9PRuW0PK+3lreo5Y4HjvMd7Y0ZJVTsQ6eqA0HEXRof11iU4oXHZdYaaMsFEe9jCr3HH2ISZ6et

In [24]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa', 'joke': [{'type': 'text', 'text': 'Why did the samosa go to therapy?\n\nBecause it was under too much pressure, felt totally **deep-fried**, and was keeping way too many **fillings** inside! \n\n***\n\n**Bonus short pun:** \nI love you **samosa** it hurts! 🥟❤️', 'extras': {'signature': 'Ep4bCpsbARFNMg8eLVy+J3Uxiyv/7wMHt5ICPRsHcR3atIGhTLdJSCCso8npax+9DCz+HCRDCYGrsNVeLCa4cz5xAS4RXxjW7wiA97eWuB67qvyCIS3fgV1z4fUKjjhdZ8J9/PLNX8YYkMGvp7QquZRnVxjpAumf2SpyhN3GxPC6FG7r2xdAllV5F8H0KhqqtAeCf9fqMhMQ2v6w8yEmcBw6VC6KgpRkR7L03zTArkG+68mfLv9RVwlfNYaH/y1g56usTMigiW4hcKIyJeN3fwqfsVRQ7dqWS7bXMrueuTh5yygr0qMOHibQyuyUC4a168mAPsofPVMPB3FIjYOOt3TpUjNLByPzBeqy1IbmdfFdFGtDhpfIKrQ9Y3hkeL9vNGfK+DznTfVJidNQRNL1N94zdvc/ICczEk0n9c84Y3sIjD5esPGHzE04Wblg/9K/X0TR2BfzGVydrKQeVmQiXa2dKoUHoyIAXyQG9ZbickD5r4Y96XWH33evo6bM0rj9am2hA5jbCBt8Bb4ZVPzvMDsN3Iyy+7TYrfZI6JtlLlnR1fjgdszeo4f83vUjNXAhvUHsE27px/cs9QcpyrWKHNU4SQdJrwxWdkx9PRuW0PK+3lreo5Y4HjvMd7Y0ZJVTsQ6eqA0HEXRof11iU4oXHZdYaaMsFE

### Fault Tolerance

In [25]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [26]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [27]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [28]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))